In [1]:
import pandas as pd
import calendar
import random
import numpy as np
from calendar import monthrange

USE_RANDOM_SEED = True
RANDOM_SEED = 42

def reset_tiebreak_rng():
    return random.Random(RANDOM_SEED) if USE_RANDOM_SEED else random.Random()

tiebreak_rng = reset_tiebreak_rng()

Important Notes:
- Matching of clerk details among different csv is done by searching up the clerk name without the rank (rank can change anytime)

In [2]:
availability = pd.read_csv("availability.csv")
points = pd.read_csv("march_points.csv")

In [3]:
points.head()

,Name,Duty
0,SHYEIKH SALEH HUSSAIN BIN MOHAMMED SULAIMAN,0.0
1,MUHAMMAD HILAL IZDIHAR BIN MOHAMAD IZZUL SHAH,2.0
2,OH EE BING,2.0
3,JEREMIAH FOO JI SHUAN,1.0
4,CHEONG KAI JIE,1.0


In [4]:
availability.head()

,Name,01/04/2026,02/04/2026,03/04/2026,04/04/2026 AM,04/04/2026 PM,05/04/2026 AM,05/04/2026 PM,06/04/2026,07/04/2026,...,24/04/2026,25/04/2026 AM,25/04/2026 PM,26/04/2026 AM,26/04/2026 PM,27/04/2026,28/04/2026,29/04/2026,30/04/2026,Availability
0,SIDDARTH SREEKUMAR CHERUBAL,1,1,1,2,1,2,1,1,1,...,1,2,1,2,1,1,1,1,1,38
1,CLARENCE ONG JUN YI,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
2,XINHANG,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
3,SONG HAOYAN,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
4,DARYL SEOW SHI YU,1,1,1,1,1,1,1,2,2,...,1,1,1,1,1,2,2,1,1,38


In [5]:
slots = availability.columns[1:-1]

## Computing Duty Clerk Point for the month

1. <b>Extended Leave.</b> If a clerk skips a month e.g. re-BMT, how will his points be calculated? Would it be the 3 most recent months?

Ans: Do not keep life-time points. If clerk skips a month, duty points will be reset to N.A.

2. <b>Procedure to assign duty. </b>
    - Assign all available duty clerks the minimum 1 duty point for the month
    - Assign the remaining duty points, based on the total duties done in the past 3 months, recent activation and "credibility ratings". Limit duty points assigned to 2
    - Let Ryan adjust the points
    - Input projected points, availabilities and preferences to algorithm

3. <b>New Clerk.</b> When a new clerk joins the roster, is there a way to decide whether he does 1 or 2 duty in the month? Should a 2nd duty be assigned to current clerks who have yet to meet their obligation first or to the new clerk first?

Ans: New Clerk does 1 duty points in the second half of the month, ideally 1 week break from the understudy period. Works out to be 1st or 2nd week: Understudy duty, followed by 3th/4th week: Actual Duty


In [6]:
# Calculate Required Duty Points for the month
YEAR = 2026
MONTH = 4
_, last_day = monthrange(YEAR, MONTH)

days = pd.date_range(start=f'2026-{MONTH:2d}-01', periods=last_day)
weekdays = np.sum(days.weekday < 5)
weekends = np.sum(days.weekday >= 5)

DUTY = weekdays * 1 + weekends * 2
print(DUTY)

38


In [7]:
planning_table = availability.copy()

# A clerk is assigned duty in a month only if they are free on at least 1 days
excluded_clerks = planning_table[planning_table["Availability"]==0]
planning_table = planning_table[planning_table["Availability"]>0]

In [8]:
excluded_clerks.head()

,Name,01/04/2026,02/04/2026,03/04/2026,04/04/2026 AM,04/04/2026 PM,05/04/2026 AM,05/04/2026 PM,06/04/2026,07/04/2026,...,24/04/2026,25/04/2026 AM,25/04/2026 PM,26/04/2026 AM,26/04/2026 PM,27/04/2026,28/04/2026,29/04/2026,30/04/2026,Availability
1,CLARENCE ONG JUN YI,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
2,XINHANG,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
3,SONG HAOYAN,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0


In [9]:
clerks = planning_table["Name"].tolist()

In [10]:
# Find duty points record for all clerk
for clerk in clerks:
    if clerk not in points["Name"].tolist():
        print(f"> No matching points found for {clerk}")
        
        # Assume the clerks are new
        # new_entry = pd.DataFrame([[clerk, 0.00]], columns=["Name", "Duty"]) # fill values for clerk not found
        # pd.concat([points, new_entry])

        # Assume the clerks ord
        planning_table = planning_table[planning_table["Name"]!=clerk]
        
points.head()

> No matching points found for JULLION THNG AIK HONG
> No matching points found for AKMAL AHMAD PUTRA BIN SAMAD
> No matching points found for MOHAMED ROSHAN AKTAR BIN AZEES ALI
> No matching points found for RAFALE


,Name,Duty
0,SHYEIKH SALEH HUSSAIN BIN MOHAMMED SULAIMAN,0.0
1,MUHAMMAD HILAL IZDIHAR BIN MOHAMAD IZZUL SHAH,2.0
2,OH EE BING,2.0
3,JEREMIAH FOO JI SHUAN,1.0
4,CHEONG KAI JIE,1.0


In [11]:
# Add duty points to planning table
planning_table = planning_table.merge(points[["Name", "Duty"]], on="Name", how="left", sort=False)

In [12]:
# Simple Projection
curr_duty = planning_table["Duty"].to_numpy()

# clerk list
clerk = planning_table["Name"]

# Assign 1 duty point per clerk
projected_duty = [1.0 if duty < 4.0 else 0.00 for duty in curr_duty]

planning_table["Projected"] = projected_duty

# Display
for name, duty in zip(clerk, projected_duty):
    print(f"{name}: {duty}")

SIDDARTH SREEKUMAR CHERUBAL: 1.0
DARYL SEOW SHI YU: 1.0
MUHAMMAD NORSYAFIQ BIN SELAMAT: 1.0
OH EE BING: 1.0
MUHAMMAD NUR HAKIM BIN SHAHARIZIN: 1.0
SURIAPRAKASH THRIUVALAKU: 1.0
FATHUL HAKIM BIN MUHAMMAD FAUZI: 1.0
EBRAHIM HAKEEM BIN AMEEHAR: 1.0
ANISH VINAAYAK SIVASANKAR: 1.0
DYLAN TAN: 1.0
SHAVIN VALEN: 1.0
CHEONG JUN KAI, HARRY: 1.0
JEREMIAH FOO JI SHUAN: 1.0
CHEONG KAI JIE: 1.0
GUO JUNQI LEO: 1.0
CHONG SIANG NATHAN: 1.0
BRYAN MARC MEHAR: 1.0
SHYEIKH SALEH HUSSAIN BIN MOHAMMED SULAIMAN: 1.0
MUHAMMAD HILAL IZDIHAR BIN MOHAMAD IZZUL SHAH: 1.0
MITCHELL LIM: 1.0
HUANG JIANHAO: 0.0
RYAN TAY RUI YANG: 1.0
ISAAC LOH KA LOK: 1.0
NICO CHUA WEE FONG: 1.0
JERRYL TAN MING JIE: 1.0
MUHAMAD RIFQY AQIL BIN MUHAMAD SHAH: 1.0
AUSTIN MAXIMILLIAN KOLBE LIM KIM: 1.0


In [13]:
planning_table.head()

,Name,01/04/2026,02/04/2026,03/04/2026,04/04/2026 AM,04/04/2026 PM,05/04/2026 AM,05/04/2026 PM,06/04/2026,07/04/2026,...,25/04/2026 PM,26/04/2026 AM,26/04/2026 PM,27/04/2026,28/04/2026,29/04/2026,30/04/2026,Availability,Duty,Projected
0,SIDDARTH SREEKUMAR CHERUBAL,1,1,1,2,1,2,1,1,1,...,1,2,1,1,1,1,1,38,0.0,1.0
1,DARYL SEOW SHI YU,1,1,1,1,1,1,1,2,2,...,1,1,1,2,2,1,1,38,1.0,1.0
2,MUHAMMAD NORSYAFIQ BIN SELAMAT,2,1,1,1,1,1,1,1,1,...,1,1,1,1,1,2,1,38,0.0,1.0
3,OH EE BING,2,2,1,1,1,0,0,0,0,...,1,1,1,2,2,2,2,29,2.0,1.0
4,MUHAMMAD NUR HAKIM BIN SHAHARIZIN,2,1,1,1,1,1,1,2,1,...,1,1,1,2,1,2,1,38,0.0,1.0


In [14]:
# Assign remaining duty points fairly
import heapq

assigned = int(planning_table["Projected"].sum())
print(f"Current Assigned Duty points is {assigned}")
tiebreak_rng = reset_tiebreak_rng()
heap =[(-point, tiebreak_rng.random(), clerk) for point, clerk in zip((4-(1+curr_duty)).tolist(), clerk)] # calculate remaining duty

heapq.heapify(heap)

for i in range(DUTY-assigned):
    neg_count, _, name = heapq.heappop(heap)
    planning_table.loc[planning_table["Name"]==name, "Projected"] += 1.00

for name, projected in zip(planning_table["Name"], planning_table["Projected"]):
    print(f"{name}: {projected}")

final = int(planning_table["Projected"].sum())
print(f"Adjusted Duty points is {final}")


Current Assigned Duty points is 26
SIDDARTH SREEKUMAR CHERUBAL: 2.0
DARYL SEOW SHI YU: 2.0
MUHAMMAD NORSYAFIQ BIN SELAMAT: 2.0
OH EE BING: 1.0
MUHAMMAD NUR HAKIM BIN SHAHARIZIN: 2.0
SURIAPRAKASH THRIUVALAKU: 1.0
FATHUL HAKIM BIN MUHAMMAD FAUZI: 1.0
EBRAHIM HAKEEM BIN AMEEHAR: 1.0
ANISH VINAAYAK SIVASANKAR: 1.0
DYLAN TAN: 1.0
SHAVIN VALEN: 2.0
CHEONG JUN KAI, HARRY: 1.0
JEREMIAH FOO JI SHUAN: 2.0
CHEONG KAI JIE: 2.0
GUO JUNQI LEO: 1.0
CHONG SIANG NATHAN: 1.0
BRYAN MARC MEHAR: 2.0
SHYEIKH SALEH HUSSAIN BIN MOHAMMED SULAIMAN: 2.0
MUHAMMAD HILAL IZDIHAR BIN MOHAMAD IZZUL SHAH: 1.0
MITCHELL LIM: 2.0
HUANG JIANHAO: 0.0
RYAN TAY RUI YANG: 1.0
ISAAC LOH KA LOK: 1.0
NICO CHUA WEE FONG: 2.0
JERRYL TAN MING JIE: 1.0
MUHAMAD RIFQY AQIL BIN MUHAMAD SHAH: 1.0
AUSTIN MAXIMILLIAN KOLBE LIM KIM: 2.0
Adjusted Duty points is 38


In [15]:
planning_table.to_csv("planning_table.csv", index=False)

## Plan for the month

- `weight_1` * preference/availability + `weight_2` * duty_count + `weight_3` * weekend

- Algorithm should space out 2 duties as much as possible

In [16]:
YEAR = 2026
MONTH = 4
DAYS = calendar.monthrange(YEAR, MONTH)[1]

In [17]:
# Update Planning Table
# 1. Duty Point this month
m, n = planning_table.shape
planning_table["Monthly Duty Points"] = [0 for _ in range(m)]

In [18]:
planning_table

,Name,01/04/2026,02/04/2026,03/04/2026,04/04/2026 AM,04/04/2026 PM,05/04/2026 AM,05/04/2026 PM,06/04/2026,07/04/2026,...,26/04/2026 AM,26/04/2026 PM,27/04/2026,28/04/2026,29/04/2026,30/04/2026,Availability,Duty,Projected,Monthly Duty Points
0,SIDDARTH SREEKUMAR CHERUBAL,1,1,1,2,1,2,1,1,1,...,2,1,1,1,1,1,38,0.0,2.0,0
1,DARYL SEOW SHI YU,1,1,1,1,1,1,1,2,2,...,1,1,2,2,1,1,38,1.0,2.0,0
2,MUHAMMAD NORSYAFIQ BIN SELAMAT,2,1,1,1,1,1,1,1,1,...,1,1,1,1,2,1,38,0.0,2.0,0
3,OH EE BING,2,2,1,1,1,0,0,0,0,...,1,1,2,2,2,2,29,2.0,1.0,0
4,MUHAMMAD NUR HAKIM BIN SHAHARIZIN,2,1,1,1,1,1,1,2,1,...,1,1,2,1,2,1,38,0.0,2.0,0
5,SURIAPRAKASH THRIUVALAKU,1,1,1,1,2,1,2,1,1,...,1,2,1,1,1,1,38,1.0,1.0,0
6,FATHUL HAKIM BIN MUHAMMAD FAUZI,1,1,2,1,1,1,1,1,1,...,1,1,1,1,1,1,38,1.0,1.0,0
7,EBRAHIM HAKEEM BIN AMEEHAR,2,1,1,0,0,0,0,2,1,...,0,0,2,1,2,1,22,2.0,1.0,0
8,ANISH VINAAYAK SIVASANKAR,2,2,2,1,1,1,1,0,0,...,1,1,2,2,2,2,32,3.0,1.0,0
9,DYLAN TAN,1,1,1,1,1,1,1,1,1,...,1,1,1,1,1,1,37,2.0,1.0,0


In [19]:
# Backtracking Algorithm
def compute_weight(date):
    weights = []
    for row in range(m):
        duty_point = 0

        availability = planning_table.loc[row, date]
        if not availability: duty_point = -1
        else: duty_point += 1

        projected = planning_table.loc[row, "Projected"]
        month = planning_table.loc[row, "Monthly Duty Points"]
        duty_count = projected - month
        if duty_count == 0: duty_point = -1
        else: duty_point += duty_count

        name = planning_table.loc[row, "Name"]
        # Keep higher duty weights first, but randomize ties instead of falling back to alphabetical order.
        heapq.heappush(weights, (-duty_point, tiebreak_rng.random(), name))
    
    return weights

def dfs(i):
    if i == len(slots):
        return True

    weights = compute_weight(slots[i])

    while weights:
        duty_points, _, name = heapq.heappop(weights)
        if -duty_points > 0:
            print(f"On {slots[i]}, {name}, {duty_points} is selected")
            row = planning_table.index[planning_table["Name"] == name].item()
            planning_table.loc[row, "Monthly Duty Points"] += 1
            if dfs(i+1):
                return True
            print("Planning Failed. Backtracking")
            planning_table.loc[row, "Monthly Duty Points"] -= 1
    
    return False

# tiebreak_rng = reset_tiebreak_rng()
# dfs(0) # Takes way too long

In [20]:
from ortools.sat.python import cp_model

# CP-SAT prototype with automatic fallback.
# Strict model:
# - exactly one clerk per slot
# - respect availability
# - match projected duties exactly
# - keep duties at least 7 days apart
# Fallback model:
# - keep availability and slot coverage hard
# - treat projected duty mismatch and short-gap violations as penalties
# - reward assigning weekend duties to clerks who marked weekend preference as 2

def parse_slot_date(slot_label):
    return pd.to_datetime(str(slot_label).split()[0], format="%d/%m/%Y")

clerk_names = planning_table["Name"].tolist()
projected_load = {
    row["Name"]: int(row["Projected"])
    for _, row in planning_table.iterrows()
}

is_available = {
    row["Name"]: {
        slot: int(row[slot])
        for slot in slots
    }
    for _, row in planning_table.iterrows()
}

slot_dates = {slot: parse_slot_date(slot) for slot in slots}
weekend_slots = [slot for slot in slots if slot_dates[slot].weekday() >= 5]
weekend_preference = {
    row["Name"]: {
        slot: 1 if slot in weekend_slots and int(row[slot]) == 2 else 0
        for slot in slots
    }
    for _, row in planning_table.iterrows()
}

def build_schedule_model(strict=True, min_gap_days=7):
    model = cp_model.CpModel()

    x = {}
    for clerk in clerk_names:
        for slot in slots:
            x[clerk, slot] = model.NewBoolVar(f"assign_{clerk}_{slot}")
            if not is_available[clerk][slot]:
                model.Add(x[clerk, slot] == 0)

    for slot in slots:
        model.Add(sum(x[clerk, slot] for clerk in clerk_names) == 1)

    total_duties = {}
    projected_diff = {}
    for clerk in clerk_names:
        total_duties[clerk] = model.NewIntVar(0, len(slots), f"total_{clerk}")
        model.Add(total_duties[clerk] == sum(x[clerk, slot] for slot in slots))
        if strict:
            model.Add(total_duties[clerk] == projected_load[clerk])
        else:
            projected_diff[clerk] = model.NewIntVar(0, len(slots), f"projected_diff_{clerk}")
            model.AddAbsEquality(projected_diff[clerk], total_duties[clerk] - projected_load[clerk])

    gap_violations = []
    for clerk in clerk_names:
        for i, slot in enumerate(slots):
            conflicting_slots = [slot]
            j = i + 1
            while j < len(slots):
                next_slot = slots[j]
                gap_days = (slot_dates[next_slot] - slot_dates[slot]).days
                if gap_days < min_gap_days:
                    conflicting_slots.append(next_slot)
                    j += 1
                else:
                    break

            if len(conflicting_slots) == 1:
                continue

            if strict:
                model.Add(sum(x[clerk, s] for s in conflicting_slots) <= 1)
            else:
                violation = model.NewIntVar(0, len(conflicting_slots) - 1, f"gap_violation_{clerk}_{i}")
                model.Add(violation >= sum(x[clerk, s] for s in conflicting_slots) - 1)
                gap_violations.append(violation)

    weekend_count = {}
    preferred_weekend_assignments = []
    for clerk in clerk_names:
        weekend_count[clerk] = model.NewIntVar(0, len(weekend_slots), f"weekend_{clerk}")
        model.Add(weekend_count[clerk] == sum(x[clerk, slot] for slot in weekend_slots))
        preferred_weekend_assignments.extend(
            weekend_preference[clerk][slot] * x[clerk, slot]
            for slot in weekend_slots
        )

    max_weekend = model.NewIntVar(0, len(weekend_slots), "max_weekend")
    min_weekend = model.NewIntVar(0, len(weekend_slots), "min_weekend")
    model.AddMaxEquality(max_weekend, list(weekend_count.values()))
    model.AddMinEquality(min_weekend, list(weekend_count.values()))
    weekend_imbalance = model.NewIntVar(0, len(weekend_slots), "weekend_imbalance")
    model.Add(weekend_imbalance == max_weekend - min_weekend)

    preferred_weekend_total = sum(preferred_weekend_assignments)

    if strict:
        model.Maximize(100 * preferred_weekend_total - 1 * weekend_imbalance)
    else:
        model.Minimize(
            100 * sum(projected_diff.values())
            + 50 * sum(gap_violations)
            + 1 * weekend_imbalance
            - 20 * preferred_weekend_total
        )

    return model, x, total_duties, weekend_count, weekend_imbalance, preferred_weekend_total

def solve_schedule(strict=True, min_gap_days=7, time_limit=10):
    model, x, total_duties, weekend_count, weekend_imbalance, preferred_weekend_total = build_schedule_model(
        strict=strict,
        min_gap_days=min_gap_days,
    )

    solver = cp_model.CpSolver()
    solver.parameters.max_time_in_seconds = time_limit
    solver.parameters.num_search_workers = 8
    status = solver.Solve(model)

    return status, solver, x, total_duties, weekend_count, weekend_imbalance, preferred_weekend_total

status, solver, x, total_duties, weekend_count, weekend_imbalance, preferred_weekend_total = solve_schedule(strict=True, min_gap_days=7)
solution_mode = "strict"

if status not in (cp_model.OPTIMAL, cp_model.FEASIBLE):
    print("Strict model infeasible. Retrying with soft projected-load and gap penalties.")
    status, solver, x, total_duties, weekend_count, weekend_imbalance, preferred_weekend_total = solve_schedule(strict=False, min_gap_days=7)
    solution_mode = "fallback"

if status not in (cp_model.OPTIMAL, cp_model.FEASIBLE):
    print("No feasible schedule found, even after fallback.")
    print("Check whether some slot has zero available clerks.")
else:
    print(f"Solved using {solution_mode} mode.")
    print(f"Weekend imbalance: {solver.Value(weekend_imbalance)}")
    print(f"Preferred weekend assignments: {solver.Value(preferred_weekend_total)}")

    schedule_rows = []
    for slot in slots:
        assigned_clerk = next(clerk for clerk in clerk_names if solver.Value(x[clerk, slot]) == 1)
        schedule_rows.append({
            "Date": slot,
            "Assigned Clerk": assigned_clerk,
            "Weekend": slot_dates[slot].weekday() >= 5,
        })

    cp_schedule = pd.DataFrame(schedule_rows)
    display(cp_schedule)

    summary_rows = []
    for clerk in clerk_names:
        summary_rows.append({
            "Name": clerk,
            "Projected": projected_load[clerk],
            "Assigned": solver.Value(total_duties[clerk]),
            "Projected Delta": solver.Value(total_duties[clerk]) - projected_load[clerk],
            "Weekend Duties": solver.Value(weekend_count[clerk]),
            "Preferred Weekend Slots": sum(weekend_preference[clerk][slot] for slot in weekend_slots),
        })

    cp_summary = pd.DataFrame(summary_rows).sort_values(
        ["Assigned", "Weekend Duties", "Name"],
        ascending=[False, False, True],
    )
    display(cp_summary)


Strict model infeasible. Retrying with soft projected-load and gap penalties.
Solved using fallback mode.
Weekend imbalance: 2
Preferred weekend assignments: 7


,Date,Assigned Clerk,Weekend
0,01/04/2026,SHYEIKH SALEH HUSSAIN BIN MOHAMMED SULAIMAN,False
1,02/04/2026,MITCHELL LIM,False
2,03/04/2026,SHAVIN VALEN,False
3,04/04/2026 AM,ANISH VINAAYAK SIVASANKAR,True
4,04/04/2026 PM,DARYL SEOW SHI YU,True
5,05/04/2026 AM,MUHAMMAD NUR HAKIM BIN SHAHARIZIN,True
6,05/04/2026 PM,MUHAMMAD NORSYAFIQ BIN SELAMAT,True
7,06/04/2026,MUHAMAD RIFQY AQIL BIN MUHAMAD SHAH,False
8,07/04/2026,AUSTIN MAXIMILLIAN KOLBE LIM KIM,False
9,08/04/2026,NICO CHUA WEE FONG,False


,Name,Projected,Assigned,Projected Delta,Weekend Duties,Preferred Weekend Slots
16,BRYAN MARC MEHAR,2,2,0,2,8
1,DARYL SEOW SHI YU,2,2,0,2,0
0,SIDDARTH SREEKUMAR CHERUBAL,2,2,0,2,8
5,SURIAPRAKASH THRIUVALAKU,1,2,1,2,8
26,AUSTIN MAXIMILLIAN KOLBE LIM KIM,2,2,0,1,0
2,MUHAMMAD NORSYAFIQ BIN SELAMAT,2,2,0,1,0
4,MUHAMMAD NUR HAKIM BIN SHAHARIZIN,2,2,0,1,0
10,SHAVIN VALEN,2,2,0,1,0
17,SHYEIKH SALEH HUSSAIN BIN MOHAMMED SULAIMAN,2,2,0,1,0
12,JEREMIAH FOO JI SHUAN,2,2,0,0,0


In [21]:
for i, row in cp_summary.iterrows():
    name = row["Name"]
    assigned = row["Assigned"]

    row = planning_table.index[planning_table["Name"] == name].item()
    planning_table.loc[row, "Monthly Duty Points"] = assigned

    

In [22]:
def check_availability_compliance(schedule_df, availability_df=planning_table, slot_col="Date", clerk_col="Assigned Clerk"):
    """Return a report showing whether each scheduled assignment is allowed by the availability table."""
    availability_lookup = availability_df.set_index("Name")
    report_rows = []

    for slot, clerk in schedule_df[[slot_col, clerk_col]].itertuples(index=False, name=None):
        if clerk not in availability_lookup.index:
            availability_value = None
            is_compliant = False
            issue = "Clerk missing from availability table"
        elif slot not in availability_lookup.columns:
            availability_value = None
            is_compliant = False
            issue = "Slot missing from availability table"
        else:
            availability_value = int(availability_lookup.at[clerk, slot])
            is_compliant = availability_value > 0
            issue = "" if is_compliant else "Assigned despite zero availability"

        report_rows.append({
            "Date": slot,
            "Assigned Clerk": clerk,
            "Availability Value": availability_value,
            "Compliant": is_compliant,
            "Issue": issue,
        })

    report_df = pd.DataFrame(report_rows)
    violations_df = report_df[~report_df["Compliant"]].copy()

    if violations_df.empty:
        print("All scheduled assignments comply with the availability table.")
    else:
        print(f"Found {len(violations_df)} availability violation(s).")
        display(violations_df)

    return report_df

# Example usage after running the CP-SAT cell:
compliance_report = check_availability_compliance(cp_schedule)
display(compliance_report)


All scheduled assignments comply with the availability table.


,Date,Assigned Clerk,Availability Value,Compliant,Issue
0,01/04/2026,SHYEIKH SALEH HUSSAIN BIN MOHAMMED SULAIMAN,2,True,
1,02/04/2026,MITCHELL LIM,1,True,
2,03/04/2026,SHAVIN VALEN,1,True,
3,04/04/2026 AM,ANISH VINAAYAK SIVASANKAR,1,True,
4,04/04/2026 PM,DARYL SEOW SHI YU,1,True,
5,05/04/2026 AM,MUHAMMAD NUR HAKIM BIN SHAHARIZIN,1,True,
6,05/04/2026 PM,MUHAMMAD NORSYAFIQ BIN SELAMAT,1,True,
7,06/04/2026,MUHAMAD RIFQY AQIL BIN MUHAMAD SHAH,1,True,
8,07/04/2026,AUSTIN MAXIMILLIAN KOLBE LIM KIM,1,True,
9,08/04/2026,NICO CHUA WEE FONG,1,True,


In [23]:
for i, row in cp_schedule.iterrows():
    name = row["Assigned Clerk"]
    date = row["Date"]

    row = planning_table.index[planning_table["Name"] == name].item()
    planning_table.loc[row, date] = 3

## Display the results nicely in a Excel File

In [24]:
from pathlib import Path
import zipfile
from html import escape

PLANNING_TABLE_EXCEL = Path("planning_table_readable.xlsx")

# Slot columns are the duty-date columns between Name and Availability.
slot_columns = list(planning_table.columns[1:planning_table.columns.get_loc("Availability")])
summary_columns = [col for col in ["Availability", "Duty", "Projected", "Monthly Duty Points"] if col in planning_table.columns]
export_columns = ["Name"] + slot_columns + summary_columns


def excel_col_name(index):
    name = ""
    while index:
        index, remainder = divmod(index - 1, 26)
        name = chr(65 + remainder) + name
    return name


def xml_cell(value, style_index=None):
    style_attr = f' s="{style_index}"' if style_index is not None else ""
    if value == "" or value is None:
        return f'<c{style_attr}/>'
    if isinstance(value, (int, float)) and not isinstance(value, bool):
        return f'<c{style_attr}><v>{value}</v></c>'
    return f'<c{style_attr} t="inlineStr"><is><t>{escape(str(value))}</t></is></c>'


def xml_row(row_number, values, styles=None):
    cells = []
    styles = styles or {}
    for col_number, value in enumerate(values, start=1):
        ref = f'{excel_col_name(col_number)}{row_number}'
        style_index = styles.get(col_number)
        cell = xml_cell(value, style_index)
        cells.append(cell.replace('<c', f'<c r="{ref}"', 1))
    return f'<row r="{row_number}">' + ''.join(cells) + '</row>'


# Style ids: 0 default, 1 header, 2 unavailable black, 3 preferred purple, 4 duty green.
rows_xml = [xml_row(1, export_columns, {i: 1 for i in range(1, len(export_columns) + 1)})]
source_values = planning_table[export_columns]
for row_number, (_, source_row) in enumerate(source_values.iterrows(), start=2):
    styles = {}
    values = []
    for col_number, col in enumerate(export_columns, start=1):
        if col in slot_columns:
            raw_value = source_row[col]
            try:
                raw_code = int(raw_value)
            except (TypeError, ValueError):
                raw_code = raw_value
            if raw_code == 0:
                styles[col_number] = 2
                values.append("")
            elif raw_code == 2:
                styles[col_number] = 3
                values.append("")
            elif raw_code == 3:
                styles[col_number] = 4
                values.append("Duty")
            else:
                values.append("")
        else:
            values.append(source_row[col])
    rows_xml.append(xml_row(row_number, values, styles))

sheet_dimension = f'A1:{excel_col_name(len(export_columns))}{len(rows_xml)}'
column_widths = [
    '<col min="1" max="1" width="34" customWidth="1"/>',
    f'<col min="2" max="{1 + len(slot_columns)}" width="13" customWidth="1"/>',
]
if summary_columns:
    column_widths.append(
        f'<col min="{2 + len(slot_columns)}" max="{len(export_columns)}" width="14" customWidth="1"/>'
    )

worksheet_xml = f'''<?xml version="1.0" encoding="UTF-8" standalone="yes"?>
<worksheet xmlns="http://schemas.openxmlformats.org/spreadsheetml/2006/main" xmlns:r="http://schemas.openxmlformats.org/officeDocument/2006/relationships">
  <dimension ref="{sheet_dimension}"/>
  <sheetViews><sheetView workbookViewId="0"><pane xSplit="1" ySplit="1" topLeftCell="B2" activePane="bottomRight" state="frozen"/><selection pane="bottomRight"/></sheetView></sheetViews>
  <cols>{''.join(column_widths)}</cols>
  <sheetData>{''.join(rows_xml)}</sheetData>
</worksheet>'''

workbook_xml = '''<?xml version="1.0" encoding="UTF-8" standalone="yes"?>
<workbook xmlns="http://schemas.openxmlformats.org/spreadsheetml/2006/main" xmlns:r="http://schemas.openxmlformats.org/officeDocument/2006/relationships"><sheets><sheet name="Planning Table" sheetId="1" r:id="rId1"/></sheets></workbook>'''

styles_xml = '''<?xml version="1.0" encoding="UTF-8" standalone="yes"?>
<styleSheet xmlns="http://schemas.openxmlformats.org/spreadsheetml/2006/main">
  <fonts count="2"><font><sz val="11"/><name val="Calibri"/></font><font><b/><sz val="11"/><name val="Calibri"/></font></fonts>
  <fills count="5"><fill><patternFill patternType="none"/></fill><fill><patternFill patternType="gray125"/></fill><fill><patternFill patternType="solid"><fgColor rgb="FF000000"/><bgColor indexed="64"/></patternFill></fill><fill><patternFill patternType="solid"><fgColor rgb="FFB4A7D6"/><bgColor indexed="64"/></patternFill></fill><fill><patternFill patternType="solid"><fgColor rgb="FF93C47D"/><bgColor indexed="64"/></patternFill></fill></fills>
  <borders count="2"><border><left/><right/><top/><bottom/><diagonal/></border><border><left style="thin"><color rgb="FFD9D9D9"/></left><right style="thin"><color rgb="FFD9D9D9"/></right><top style="thin"><color rgb="FFD9D9D9"/></top><bottom style="thin"><color rgb="FFD9D9D9"/></bottom><diagonal/></border></borders>
  <cellStyleXfs count="1"><xf numFmtId="0" fontId="0" fillId="0" borderId="0"/></cellStyleXfs>
  <cellXfs count="5"><xf numFmtId="0" fontId="0" fillId="0" borderId="1" xfId="0"/><xf numFmtId="0" fontId="1" fillId="0" borderId="1" xfId="0" applyFont="1"/><xf numFmtId="0" fontId="0" fillId="2" borderId="1" xfId="0" applyFill="1"/><xf numFmtId="0" fontId="0" fillId="3" borderId="1" xfId="0" applyFill="1"/><xf numFmtId="0" fontId="1" fillId="4" borderId="1" xfId="0" applyFont="1" applyFill="1"/></cellXfs>
  <cellStyles count="1"><cellStyle name="Normal" xfId="0" builtinId="0"/></cellStyles>
</styleSheet>'''

content_types_xml = '''<?xml version="1.0" encoding="UTF-8" standalone="yes"?>
<Types xmlns="http://schemas.openxmlformats.org/package/2006/content-types"><Default Extension="rels" ContentType="application/vnd.openxmlformats-package.relationships+xml"/><Default Extension="xml" ContentType="application/xml"/><Override PartName="/xl/workbook.xml" ContentType="application/vnd.openxmlformats-officedocument.spreadsheetml.sheet.main+xml"/><Override PartName="/xl/worksheets/sheet1.xml" ContentType="application/vnd.openxmlformats-officedocument.spreadsheetml.worksheet+xml"/><Override PartName="/xl/styles.xml" ContentType="application/vnd.openxmlformats-officedocument.spreadsheetml.styles+xml"/></Types>'''

root_rels_xml = '''<?xml version="1.0" encoding="UTF-8" standalone="yes"?>
<Relationships xmlns="http://schemas.openxmlformats.org/package/2006/relationships"><Relationship Id="rId1" Type="http://schemas.openxmlformats.org/officeDocument/2006/relationships/officeDocument" Target="xl/workbook.xml"/></Relationships>'''

workbook_rels_xml = '''<?xml version="1.0" encoding="UTF-8" standalone="yes"?>
<Relationships xmlns="http://schemas.openxmlformats.org/package/2006/relationships"><Relationship Id="rId1" Type="http://schemas.openxmlformats.org/officeDocument/2006/relationships/worksheet" Target="worksheets/sheet1.xml"/><Relationship Id="rId2" Type="http://schemas.openxmlformats.org/officeDocument/2006/relationships/styles" Target="styles.xml"/></Relationships>'''

with zipfile.ZipFile(PLANNING_TABLE_EXCEL, "w", compression=zipfile.ZIP_DEFLATED) as workbook:
    workbook.writestr("[Content_Types].xml", content_types_xml)
    workbook.writestr("_rels/.rels", root_rels_xml)
    workbook.writestr("xl/workbook.xml", workbook_xml)
    workbook.writestr("xl/_rels/workbook.xml.rels", workbook_rels_xml)
    workbook.writestr("xl/worksheets/sheet1.xml", worksheet_xml)
    workbook.writestr("xl/styles.xml", styles_xml)

print(f"Wrote {PLANNING_TABLE_EXCEL.resolve()}")


Wrote /Users/harry/Desktop/Projects/Totally Fair Scheduler/planning_table_readable.xlsx


# Reserve 1 Planning

In [25]:
for i, row in cp_schedule.iterrows():
    name = row["Assigned Clerk"]
    date = row["Date"]

    row = planning_table.index[planning_table["Name"] == name].item()
    
    planning_table.loc[row, date] = 0

In [26]:
planning_table

,Name,01/04/2026,02/04/2026,03/04/2026,04/04/2026 AM,04/04/2026 PM,05/04/2026 AM,05/04/2026 PM,06/04/2026,07/04/2026,...,26/04/2026 AM,26/04/2026 PM,27/04/2026,28/04/2026,29/04/2026,30/04/2026,Availability,Duty,Projected,Monthly Duty Points
0,SIDDARTH SREEKUMAR CHERUBAL,1,1,1,2,1,2,1,1,1,...,0,1,1,1,1,1,38,0.0,2.0,2
1,DARYL SEOW SHI YU,1,1,1,1,0,1,1,2,2,...,1,1,2,2,1,1,38,1.0,2.0,2
2,MUHAMMAD NORSYAFIQ BIN SELAMAT,2,1,1,1,1,1,0,1,1,...,1,1,1,1,2,1,38,0.0,2.0,2
3,OH EE BING,2,2,1,1,1,0,0,0,0,...,1,1,2,2,0,2,29,2.0,1.0,1
4,MUHAMMAD NUR HAKIM BIN SHAHARIZIN,2,1,1,1,1,0,1,2,1,...,1,1,2,1,2,1,38,0.0,2.0,2
5,SURIAPRAKASH THRIUVALAKU,1,1,1,1,2,1,2,1,1,...,1,2,1,1,1,1,38,1.0,1.0,2
6,FATHUL HAKIM BIN MUHAMMAD FAUZI,1,1,2,1,1,1,1,1,1,...,1,1,1,1,1,1,38,1.0,1.0,1
7,EBRAHIM HAKEEM BIN AMEEHAR,2,1,1,0,0,0,0,2,1,...,0,0,2,1,2,1,22,2.0,1.0,1
8,ANISH VINAAYAK SIVASANKAR,2,2,2,0,1,1,1,0,0,...,1,1,2,2,2,2,32,3.0,1.0,1
9,DYLAN TAN,1,1,1,1,1,1,1,1,1,...,1,1,1,1,1,1,37,2.0,1.0,1


In [27]:
from ortools.sat.python import cp_model

# CP-SAT prototype with automatic fallback.
# Strict model:
# - exactly one clerk per slot
# - respect availability
# - match projected duties exactly
# - keep duties at least 7 days apart
# Fallback model:
# - keep availability and slot coverage hard
# - treat projected duty mismatch and short-gap violations as penalties
# - reward assigning weekend duties to clerks who marked weekend preference as 2

def parse_slot_date(slot_label):
    return pd.to_datetime(str(slot_label).split()[0], format="%d/%m/%Y")

clerk_names = planning_table["Name"].tolist()
projected_load = {
    row["Name"]: int(row["Projected"])
    for _, row in planning_table.iterrows()
}

is_available = {
    row["Name"]: {
        slot: int(row[slot])
        for slot in slots
    }
    for _, row in planning_table.iterrows()
}

slot_dates = {slot: parse_slot_date(slot) for slot in slots}
weekend_slots = [slot for slot in slots if slot_dates[slot].weekday() >= 5]
weekend_preference = {
    row["Name"]: {
        slot: 1 if slot in weekend_slots and int(row[slot]) == 2 else 0
        for slot in slots
    }
    for _, row in planning_table.iterrows()
}

def build_schedule_model(strict=True, min_gap_days=7):
    model = cp_model.CpModel()

    x = {}
    for clerk in clerk_names:
        for slot in slots:
            x[clerk, slot] = model.NewBoolVar(f"assign_{clerk}_{slot}")
            if not is_available[clerk][slot]:
                model.Add(x[clerk, slot] == 0)

    for slot in slots:
        model.Add(sum(x[clerk, slot] for clerk in clerk_names) == 1)

    total_duties = {}
    projected_diff = {}
    for clerk in clerk_names:
        total_duties[clerk] = model.NewIntVar(0, len(slots), f"total_{clerk}")
        model.Add(total_duties[clerk] == sum(x[clerk, slot] for slot in slots))
        if strict:
            model.Add(total_duties[clerk] == projected_load[clerk])
        else:
            projected_diff[clerk] = model.NewIntVar(0, len(slots), f"projected_diff_{clerk}")
            model.AddAbsEquality(projected_diff[clerk], total_duties[clerk] - projected_load[clerk])

    gap_violations = []
    for clerk in clerk_names:
        for i, slot in enumerate(slots):
            conflicting_slots = [slot]
            j = i + 1
            while j < len(slots):
                next_slot = slots[j]
                gap_days = (slot_dates[next_slot] - slot_dates[slot]).days
                if gap_days < min_gap_days:
                    conflicting_slots.append(next_slot)
                    j += 1
                else:
                    break

            if len(conflicting_slots) == 1:
                continue

            if strict:
                model.Add(sum(x[clerk, s] for s in conflicting_slots) <= 1)
            else:
                violation = model.NewIntVar(0, len(conflicting_slots) - 1, f"gap_violation_{clerk}_{i}")
                model.Add(violation >= sum(x[clerk, s] for s in conflicting_slots) - 1)
                gap_violations.append(violation)

    weekend_count = {}
    preferred_weekend_assignments = []
    for clerk in clerk_names:
        weekend_count[clerk] = model.NewIntVar(0, len(weekend_slots), f"weekend_{clerk}")
        model.Add(weekend_count[clerk] == sum(x[clerk, slot] for slot in weekend_slots))
        preferred_weekend_assignments.extend(
            weekend_preference[clerk][slot] * x[clerk, slot]
            for slot in weekend_slots
        )

    max_weekend = model.NewIntVar(0, len(weekend_slots), "max_weekend")
    min_weekend = model.NewIntVar(0, len(weekend_slots), "min_weekend")
    model.AddMaxEquality(max_weekend, list(weekend_count.values()))
    model.AddMinEquality(min_weekend, list(weekend_count.values()))
    weekend_imbalance = model.NewIntVar(0, len(weekend_slots), "weekend_imbalance")
    model.Add(weekend_imbalance == max_weekend - min_weekend)

    preferred_weekend_total = sum(preferred_weekend_assignments)

    if strict:
        model.Maximize(100 * preferred_weekend_total - 1 * weekend_imbalance)
    else:
        model.Minimize(
            100 * sum(projected_diff.values())
            + 50 * sum(gap_violations)
            + 1 * weekend_imbalance
            - 20 * preferred_weekend_total
        )

    return model, x, total_duties, weekend_count, weekend_imbalance, preferred_weekend_total

def solve_schedule(strict=True, min_gap_days=7, time_limit=10):
    model, x, total_duties, weekend_count, weekend_imbalance, preferred_weekend_total = build_schedule_model(
        strict=strict,
        min_gap_days=min_gap_days,
    )

    solver = cp_model.CpSolver()
    solver.parameters.max_time_in_seconds = time_limit
    solver.parameters.num_search_workers = 8
    status = solver.Solve(model)

    return status, solver, x, total_duties, weekend_count, weekend_imbalance, preferred_weekend_total

status, solver, x, total_duties, weekend_count, weekend_imbalance, preferred_weekend_total = solve_schedule(strict=True, min_gap_days=7)
solution_mode = "strict"

if status not in (cp_model.OPTIMAL, cp_model.FEASIBLE):
    print("Strict model infeasible. Retrying with soft projected-load and gap penalties.")
    status, solver, x, total_duties, weekend_count, weekend_imbalance, preferred_weekend_total = solve_schedule(strict=False, min_gap_days=7)
    solution_mode = "fallback"

if status not in (cp_model.OPTIMAL, cp_model.FEASIBLE):
    print("No feasible schedule found, even after fallback.")
    print("Check whether some slot has zero available clerks.")
else:
    print(f"Solved using {solution_mode} mode.")
    print(f"Weekend imbalance: {solver.Value(weekend_imbalance)}")
    print(f"Preferred weekend assignments: {solver.Value(preferred_weekend_total)}")

    schedule_rows = []
    for slot in slots:
        assigned_clerk = next(clerk for clerk in clerk_names if solver.Value(x[clerk, slot]) == 1)
        schedule_rows.append({
            "Date": slot,
            "Assigned Clerk": assigned_clerk,
            "Weekend": slot_dates[slot].weekday() >= 5,
        })

    cp_reserve1_schedule = pd.DataFrame(schedule_rows)
    display(cp_reserve1_schedule)

    reserve1_summary_rows = []
    for clerk in clerk_names:
        reserve1_summary_rows.append({
            "Name": clerk,
            "Projected": projected_load[clerk],
            "Assigned": solver.Value(total_duties[clerk]),
            "Projected Delta": solver.Value(total_duties[clerk]) - projected_load[clerk],
            "Weekend Duties": solver.Value(weekend_count[clerk]),
            "Preferred Weekend Slots": sum(weekend_preference[clerk][slot] for slot in weekend_slots),
        })

    cp_reserve1_summary = pd.DataFrame(reserve1_summary_rows).sort_values(
        ["Assigned", "Weekend Duties", "Name"],
        ascending=[False, False, True],
    )
    display(cp_reserve1_summary)


Strict model infeasible. Retrying with soft projected-load and gap penalties.
Solved using fallback mode.
Weekend imbalance: 2
Preferred weekend assignments: 7


,Date,Assigned Clerk,Weekend
0,01/04/2026,JEREMIAH FOO JI SHUAN,False
1,02/04/2026,MUHAMMAD NUR HAKIM BIN SHAHARIZIN,False
2,03/04/2026,SHYEIKH SALEH HUSSAIN BIN MOHAMMED SULAIMAN,False
3,04/04/2026 AM,BRYAN MARC MEHAR,True
4,04/04/2026 PM,SHAVIN VALEN,True
5,05/04/2026 AM,SIDDARTH SREEKUMAR CHERUBAL,True
6,05/04/2026 PM,MUHAMMAD HILAL IZDIHAR BIN MOHAMAD IZZUL SHAH,True
7,06/04/2026,DARYL SEOW SHI YU,False
8,07/04/2026,EBRAHIM HAKEEM BIN AMEEHAR,False
9,08/04/2026,MUHAMMAD NORSYAFIQ BIN SELAMAT,False


,Name,Projected,Assigned,Projected Delta,Weekend Duties,Preferred Weekend Slots
16,BRYAN MARC MEHAR,2,2,0,2,6
19,MITCHELL LIM,2,2,0,2,0
0,SIDDARTH SREEKUMAR CHERUBAL,2,2,0,2,6
5,SURIAPRAKASH THRIUVALAKU,1,2,1,2,6
12,JEREMIAH FOO JI SHUAN,2,2,0,1,0
4,MUHAMMAD NUR HAKIM BIN SHAHARIZIN,2,2,0,1,0
23,NICO CHUA WEE FONG,2,2,0,1,0
10,SHAVIN VALEN,2,2,0,1,0
17,SHYEIKH SALEH HUSSAIN BIN MOHAMMED SULAIMAN,2,2,0,1,0
26,AUSTIN MAXIMILLIAN KOLBE LIM KIM,2,2,0,0,0


In [28]:
reserve1_compliance_report = check_availability_compliance(cp_reserve1_schedule)
display(reserve1_compliance_report)

All scheduled assignments comply with the availability table.


,Date,Assigned Clerk,Availability Value,Compliant,Issue
0,01/04/2026,JEREMIAH FOO JI SHUAN,2,True,
1,02/04/2026,MUHAMMAD NUR HAKIM BIN SHAHARIZIN,1,True,
2,03/04/2026,SHYEIKH SALEH HUSSAIN BIN MOHAMMED SULAIMAN,1,True,
3,04/04/2026 AM,BRYAN MARC MEHAR,2,True,
4,04/04/2026 PM,SHAVIN VALEN,1,True,
5,05/04/2026 AM,SIDDARTH SREEKUMAR CHERUBAL,2,True,
6,05/04/2026 PM,MUHAMMAD HILAL IZDIHAR BIN MOHAMAD IZZUL SHAH,1,True,
7,06/04/2026,DARYL SEOW SHI YU,2,True,
8,07/04/2026,EBRAHIM HAKEEM BIN AMEEHAR,1,True,
9,08/04/2026,MUHAMMAD NORSYAFIQ BIN SELAMAT,2,True,


In [29]:
for i, row in cp_reserve1_schedule.iterrows():
    name = row["Assigned Clerk"]
    date = row["Date"]

    row = planning_table.index[planning_table["Name"] == name].item()
    
    planning_table.loc[row, date] = 0

## Reserve 2 Planning